In [1]:
import json
import os
import pandas as pd
import matplotlib.pyplot as plt

import warnings

In [2]:
warnings.filterwarnings('ignore')

In [4]:
storm_outages = pd.read_parquet('../data/interim/storm_outages_2014_2023.parquet')

In [17]:
# Corroborar que solo haya dos valores
storm_outages.groupby('episode_fips_id').storm_caused_outage.mean().value_counts()

storm_caused_outage
0.0    388113
1.0     24346
Name: count, dtype: int64

In [22]:
episode_fips_id_value_counts = storm_outages.episode_fips_id.value_counts()
episode_fips_id_value_counts_gt1 = episode_fips_id_value_counts[episode_fips_id_value_counts > 1]

In [27]:
storm_outages[storm_outages.episode_fips_id.isin(episode_fips_id_value_counts_gt1.index)].storm_caused_outage.unique()
# Una misma tormeta puede causar varios outages en el mismo county los cuales no están relacionados entre sí. 
storm_outages[storm_outages.episode_fips_id=='107733_06037'].head()

,EPISODE_ID,fips_code_id,episode_description,begin_datetime,end_datetime,storm_duration,episode_fips_id,storm_caused_outage,outage_index_id,outage_start_minus_storm_start,outage_end_minus_storm_end,outage_start_minus_storm_end,outage_end_minus_storm_start,outage_duration,run_start_time_min,run_start_time_max
92952,107733,06037,Due to very dry vegetation and continued hot a...,2016-07-09 11:04:00,2016-07-31 23:59:00,540.916667,107733_06037,1.0,06037__0227,1.007639,-21.124306,-21.530556,1.413889,0.406250,2016-07-10 11:15:00,2016-07-10 21:00:00
92953,107733,06037,Due to very dry vegetation and continued hot a...,2016-07-09 11:04:00,2016-07-31 23:59:00,540.916667,107733_06037,1.0,06037__0228,3.413889,-19.113889,-19.124306,3.424306,0.010417,2016-07-12 21:00:00,2016-07-12 21:15:00
92954,107733,06037,Due to very dry vegetation and continued hot a...,2016-07-09 11:04:00,2016-07-31 23:59:00,540.916667,107733_06037,1.0,06037__0229,4.257639,-18.270139,-18.280556,4.268056,0.010417,2016-07-13 17:15:00,2016-07-13 17:30:00
92955,107733,06037,Due to very dry vegetation and continued hot a...,2016-07-09 11:04:00,2016-07-31 23:59:00,540.916667,107733_06037,1.0,06037__0230,6.278472,-15.457639,-16.259722,7.080556,0.802083,2016-07-15 17:45:00,2016-07-16 13:00:00
92956,107733,06037,Due to very dry vegetation and continued hot a...,2016-07-09 11:04:00,2016-07-31 23:59:00,540.916667,107733_06037,1.0,06037__0231,8.382639,-13.790972,-14.155556,8.747222,0.364583,2016-07-17 20:15:00,2016-07-18 05:00:00


In [46]:
# Simplification, we weill keep the smallest outage_date, and the longest outage_date (all the sepparated outage will be considered as just one big outage.)
storm_outages_g = storm_outages.groupby('episode_fips_id').agg(
    EPISODE_ID=('EPISODE_ID','first'),
    fips_code_id=('fips_code_id','first'),
    episode_description=('episode_description','first'),
    begin_datetime=('begin_datetime','first'),
    end_datetime=('end_datetime','first'),
    storm_duration=('storm_duration','first'),
    #episode_fips_id=('episode_fips_id','first'),
    storm_caused_outage=('storm_caused_outage','max'),
    outage_index_id=('outage_index_id','first'),
    outage_duration =('outage_duration', 'sum'),
    run_start_time_min=('run_start_time_min','min'),
    run_start_time_max=('run_start_time_max','max'),
).reset_index()


In [47]:
storm_outages_g

,episode_fips_id,EPISODE_ID,fips_code_id,episode_description,begin_datetime,end_datetime,storm_duration,storm_caused_outage,outage_index_id,outage_duration,run_start_time_min,run_start_time_max
0,100001_51139,100001,51139,High pressure sprawled over the Mid-Atlantic r...,2015-09-22 23:35:00,2015-09-23 08:00:00,8.416667,0.0,None,0.0,NaT,NaT
1,100001_51165,100001,51165,High pressure sprawled over the Mid-Atlantic r...,2015-09-22 23:35:00,2015-09-23 08:00:00,8.416667,0.0,None,0.0,NaT,NaT
2,100001_51171,100001,51171,High pressure sprawled over the Mid-Atlantic r...,2015-09-22 23:35:00,2015-09-23 08:00:00,8.416667,0.0,None,0.0,NaT,NaT
3,100001_51187,100001,51187,High pressure sprawled over the Mid-Atlantic r...,2015-09-22 23:35:00,2015-09-23 08:00:00,8.416667,0.0,None,0.0,NaT,NaT
4,100002_54023,100002,54023,High pressure sprawled over the Mid-Atlantic r...,2015-09-24 04:30:00,2015-09-24 09:35:00,5.083333,0.0,None,0.0,NaT,NaT
...,...,...,...,...,...,...,...,...,...,...,...,...
412454,99997_88538,99997,88538,Scattered showers and isolated thunderstorms f...,2015-08-11 14:12:00,2015-08-11 17:30:00,3.300000,0.0,None,0.0,NaT,NaT
412455,99998_88536,99998,88536,Strong moisture advection over the Mid-Atlanti...,2015-08-20 19:34:00,2015-08-20 21:36:00,2.033333,0.0,None,0.0,NaT,NaT
412456,99998_88537,99998,88537,Strong moisture advection over the Mid-Atlanti...,2015-08-20 19:34:00,2015-08-20 21:36:00,2.033333,0.0,None,0.0,NaT,NaT
412457,99999_88531,99999,88531,"A cold front moved through the Mid-Atlantic, t...",2015-08-24 18:49:00,2015-08-24 19:51:00,1.033333,0.0,None,0.0,NaT,NaT


In [48]:
meteorological_path = '../data/raw/meteorological'
meteorological_files = os.listdir(meteorological_path)

In [53]:
def get_data_with_response_variable(met_file):
    try:
    #if True:
        #met_file = meteorological_files[0]
        episode_fips_id = met_file.split('.')[0]
        #  Read data
        path = os.path.join(meteorological_path, met_file)
        
        with open(path, 'r') as f:
            data = json.load(f)
        # Make a dataframe from information
        data = pd.DataFrame(data.get('properties').get('parameter'))
        data.index.rename('time', inplace=True)
        data.reset_index(inplace=True)
        data['episode_fips_id'] = episode_fips_id
        data['meteorological_current_datetime_val'] = pd.to_datetime(
            data['time'].apply(lambda x: x[0:4]) + 
            '-' + 
            data['time'].apply(lambda x: x[4:6]) +
            '-' + 
            data['time'].apply(lambda x: x[6:8]) +
            ' ' +
            data['time'].apply(lambda x: x[8:10]) +
            ':00:00'
        )
        # Modification to change the hour treshold.
        data['meteorological_nextHour_datetime_val'] = data['meteorological_current_datetime_val'] + pd.Timedelta(value=1, unit='hours')
        # Get the response var dataframe
        storm_outage_episode = storm_outages_g[storm_outages_g.episode_fips_id == episode_fips_id]
        meaning_dict = {
            'begin_datetime': 'storm_start', 
            'end_datetime': 'storm_end', 
            'run_start_time_min': 'outage_start', 
            'run_start_time_max': 'outage_end'
        }
        storm_outage_episode.rename(columns=meaning_dict, inplace=True)
        storm_outage_episode.drop_duplicates(['episode_fips_id'], keep='first', inplace=True)
        storm_outage_episode_columns = [
            'episode_fips_id', 
            'storm_start', 
            #'storm_end',
            'outage_start',
            'outage_end'
        ]
        # Join information
        data_rv = data.merge(
            storm_outage_episode[storm_outage_episode_columns],
            on='episode_fips_id',
            how='left'
        )
        data_rv['outage_start_rounded'] = data_rv.outage_start.dt.round('H')
        data_rv['outage_end_rounded'] = data_rv.outage_end.dt.round('H')
        data_rv = data_rv[data_rv.meteorological_nextHour_datetime_val <= data_rv.outage_start_rounded]
        data_rv = data_rv[data_rv.meteorological_nextHour_datetime_val >= data_rv.storm_start]
        data_rv.loc[data_rv.meteorological_nextHour_datetime_val == data_rv.outage_start_rounded, 'outage_in_an_hour'] = 1
        data_rv.loc[data_rv.meteorological_nextHour_datetime_val != data_rv.outage_start_rounded, 'outage_in_an_hour'] = 0
        data_rv['episode_fips_time_id'] = data_rv.episode_fips_id + '_' + data_rv.time.astype(str)
        data_rv.set_index('episode_fips_time_id', inplace=True)
        return data_rv
    except:
        return None


In [54]:
#data_test, response_test = get_data_with_response_variable('164119_06029.json')

In [55]:
#response_test

In [57]:
data_rvs = []
for met_file in meteorological_files:
    data_rvs.append(get_data_with_response_variable(met_file))


In [58]:
cleaned_data_rvs = [x for x in data_rvs if x is not None]

In [59]:
all_data = pd.concat(cleaned_data_rvs)

In [60]:
all_data.outage_in_an_hour.sum()

np.float64(4882.0)

In [61]:
all_data.outage_in_an_hour.mean()

np.float64(0.021575516517511877)

In [62]:
pd.set_option('display.max_columns', 100)
all_data

,time,T2M,ALLSKY_SFC_SW_DWN,PS,WS50M,ALLSKY_SFC_LW_DWN,WD50M,PRECTOTCORR,GWETPROF,CLRSKY_SFC_SW_DWN,QV10M,RHOA,T10M,TO3,TQV,Z0M,TOA_SW_DWN,RH2M,WS2M,CLRSKY_SFC_LW_DWN,DISPH,episode_fips_id,meteorological_current_datetime_val,meteorological_nextHour_datetime_val,storm_start,outage_start,outage_end,outage_start_rounded,outage_end_rounded,outage_in_an_hour
episode_fips_time_id,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
100009_55087_2015081413,2015081413,31.35,639.47,98.56,6.65,419.52,290.9,7.12,0.61,783.12,14.71,1.12,30.76,312.70,34.17,0.08,1071.88,53.04,3.85,404.27,0.38,100009_55087,2015-08-14 13:00:00,2015-08-14 14:00:00,2015-08-14 13:30:00,2015-08-14 17:15:00,2015-08-14 17:30:00,2015-08-14 17:00:00,2015-08-14 18:00:00,0.0
100009_55087_2015081414,2015081414,31.11,343.15,98.54,6.78,446.12,296.1,10.01,0.61,680.40,15.14,1.12,30.65,311.35,34.49,0.08,951.17,55.00,3.85,403.17,0.38,100009_55087,2015-08-14 14:00:00,2015-08-14 15:00:00,2015-08-14 13:30:00,2015-08-14 17:15:00,2015-08-14 17:30:00,2015-08-14 17:00:00,2015-08-14 18:00:00,0.0
100009_55087_2015081415,2015081415,30.47,123.62,98.54,6.83,429.02,299.4,11.55,0.61,536.53,15.37,1.12,30.17,309.48,34.42,0.08,781.22,57.65,3.78,402.08,0.38,100009_55087,2015-08-14 15:00:00,2015-08-14 16:00:00,2015-08-14 13:30:00,2015-08-14 17:15:00,2015-08-14 17:30:00,2015-08-14 17:00:00,2015-08-14 18:00:00,0.0
100009_55087_2015081416,2015081416,29.52,55.92,98.56,6.46,433.27,300.7,13.07,0.61,364.85,15.63,1.13,29.42,307.94,33.63,0.08,573.58,61.57,3.42,399.38,0.38,100009_55087,2015-08-14 16:00:00,2015-08-14 17:00:00,2015-08-14 13:30:00,2015-08-14 17:15:00,2015-08-14 17:30:00,2015-08-14 17:00:00,2015-08-14 18:00:00,1.0
100103_45007_2015080612,2015080612,34.52,771.45,98.51,4.36,427.27,239.7,1.68,0.51,905.12,14.07,1.12,33.04,307.98,42.68,1.49,1255.90,41.11,1.02,402.65,7.56,100103_45007,2015-08-06 12:00:00,2015-08-06 13:00:00,2015-08-06 12:59:00,2015-08-06 22:00:00,2015-08-06 22:30:00,2015-08-06 22:00:00,2015-08-06 22:00:00,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99962_34029_2015100109,2015100109,13.75,97.88,101.38,10.18,381.15,34.7,13.91,0.43,515.70,7.68,1.23,13.58,282.39,44.37,0.43,811.95,81.55,5.38,342.30,2.16,99962_34029,2015-10-01 09:00:00,2015-10-01 10:00:00,2015-10-01 00:00:00,2015-10-01 13:45:00,2015-10-11 14:00:00,2015-10-01 14:00:00,2015-10-11 14:00:00,0.0
99962_34029_2015100110,2015100110,14.11,139.27,101.38,10.57,372.33,36.8,19.36,0.43,626.17,7.76,1.23,13.90,280.92,44.23,0.43,935.03,80.67,5.57,342.95,2.16,99962_34029,2015-10-01 10:00:00,2015-10-01 11:00:00,2015-10-01 00:00:00,2015-10-01 13:45:00,2015-10-11 14:00:00,2015-10-01 14:00:00,2015-10-11 14:00:00,0.0
99962_34029_2015100111,2015100111,14.66,103.60,101.37,10.97,385.25,38.5,27.39,0.43,670.97,7.85,1.23,14.41,279.62,43.61,0.43,991.05,78.90,5.78,343.30,2.16,99962_34029,2015-10-01 11:00:00,2015-10-01 12:00:00,2015-10-01 00:00:00,2015-10-01 13:45:00,2015-10-11 14:00:00,2015-10-01 14:00:00,2015-10-11 14:00:00,0.0


In [64]:
all_data.loc['164119_06029_2021120823']

time                                             2021120823
T2M                                                    8.91
ALLSKY_SFC_SW_DWN                                       0.0
PS                                                    92.91
WS50M                                                  1.78
ALLSKY_SFC_LW_DWN                                    302.33
WD50M                                                 233.9
PRECTOTCORR                                            0.84
GWETPROF                                               0.55
CLRSKY_SFC_SW_DWN                                       0.0
QV10M                                                  6.95
RHOA                                                   1.14
T10M                                                   9.12
TO3                                                  283.58
TQV                                                   20.32
Z0M                                                     0.1
TOA_SW_DWN                              

In [65]:
all_data.drop(
    ['meteorological_nextHour_datetime_val', 
     'storm_start', 'outage_start', 'outage_end', 'outage_start_rounded', 'outage_end_rounded'], axis=1
).to_parquet('../data/interim/meteorological_data_with_outages.parquet')